## Embeddings

Embeddings are a numerical representation of a text. Basically a list of floating point numbers. 
`Why?`
Machine learnign models dont use text for training or any of the calculations. They need numbers. Embeddings provides us with that feature.  
these embeddings are used for semantic searches meaning 2 embedding swith relatively same value have similar semantic meaning which means for example `COW` and `Animal` will be close to each other int he vector space.  
But `COW` and `Airplane` will be dar away from each other since they dont have semantic meaning.  
There are many models which help in converting the test into embeddings and we can use the apis to convert the text into embeddings with couple of lines of code.


In [7]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

import pandas as pd
df = pd.read_csv('words.csv')
df = df.sample(frac=1)
df

,text
24,sandwich
13,raccoon
21,milk
38,green
30,pizza
19,lizard
39,gray
6,rabbit
37,guinea pig
10,cappuccino


In [8]:
from openai import OpenAI
def get_embedding(text, model='text-embedding-3-small'):
    client = OpenAI()
    text = text.replace('\n', ' ')
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding
    

In [9]:
df['embedding'] = df['text'].apply(lambda x: get_embedding(x))

In [10]:
df

,text,embedding
24,sandwich,"[-0.00571441650390625, -0.048858642578125, -0...."
13,raccoon,"[0.05898793041706085, -0.0010472146095708013, ..."
21,milk,"[0.039642333984375, -0.004215240478515625, -0...."
38,green,"[0.006280624307692051, -0.0011062989942729473,..."
30,pizza,"[-0.010169987566769123, -0.028292447328567505,..."
19,lizard,"[0.01209514494985342, -0.012445430271327496, -..."
39,gray,"[0.004851989448070526, -0.02022680640220642, -..."
6,rabbit,"[0.006238460540771484, -0.01577582396566868, 0..."
37,guinea pig,"[0.0074920654296875, -0.0266571044921875, 0.00..."
10,cappuccino,"[-0.0248565673828125, -0.0309295654296875, -0...."


In [11]:
df.to_csv('words_embeddings.csv', index=False)

### Estimating embedding costs with tiktoken

In [12]:
import tiktoken
import pandas as pd

df = pd.read_csv('words_embeddings.csv')
df.head()

,text,embedding
0,sandwich,"[-0.00571441650390625, -0.048858642578125, -0...."
1,raccoon,"[0.05898793041706085, -0.0010472146095708013, ..."
2,milk,"[0.039642333984375, -0.004215240478515625, -0...."
3,green,"[0.006280624307692051, -0.0011062989942729473,..."
4,pizza,"[-0.010169987566769123, -0.028292447328567505,..."


In [13]:
words = list(df['text'])
words[:10]

['sandwich',
 'raccoon',
 'milk',
 'green',
 'pizza',
 'lizard',
 'gray',
 'rabbit',
 'guinea pig',
 'cappuccino']

In [15]:
enc = tiktoken.encoding_for_model('text-embedding-3-small')
total_tokens = sum([len(enc.encode(word)) for word in words])
print(total_tokens)

62


In [17]:
cost_per_token = 0.02/1000000
estimated_cost = total_tokens * cost_per_token
print(f'Cost of embedding {total_tokens} tokens: ${estimated_cost:0.10f}')

Cost of embedding 62 tokens: $0.0000012400


### Performing Semantic Searches

In [19]:
import pandas as pd
import numpy as np
df = pd.read_csv('words_embeddings.csv')
df.head()

,text,embedding
0,sandwich,"[-0.00571441650390625, -0.048858642578125, -0...."
1,raccoon,"[0.05898793041706085, -0.0010472146095708013, ..."
2,milk,"[0.039642333984375, -0.004215240478515625, -0...."
3,green,"[0.006280624307692051, -0.0011062989942729473,..."
4,pizza,"[-0.010169987566769123, -0.028292447328567505,..."


In [20]:
df['embedding'] = df['embedding'].apply(eval).apply(np.array)

In [23]:
search_term = 'red'
search_term_vector = get_embedding(search_term)
def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)
    magnitude_a = np.linalg.norm(vector_a, 2) # L2 norm
    magnitude_b = np.linalg.norm(vector_b, 2) # L2 norm
    
    if magnitude_a == 0 or magnitude_b == 0:
        return 0 # Or handle as appropriate for your application
        
    cosine_similarity = dot_product / (magnitude_a * magnitude_b)
    return cosine_similarity

df['similarities'] = df['embedding'].apply(lambda x: cosine_similarity(x, search_term_vector))
df.sort_values('similarities', ascending=False).head(10)

,text,embedding,similarities
37,red,"[-0.02211996167898178, -0.010933708399534225, ...",0.999999
36,blue,"[-0.0010865783551707864, -0.016484010964632034...",0.641060
11,purple,"[0.0247344970703125, -0.03472900390625, -0.026...",0.565471
22,white,"[0.0032487320713698864, -0.02826567552983761, ...",0.547506
24,orange,"[-0.0258636474609375, -0.005527496337890625, -...",0.544711
12,deer,"[0.060791015625, -0.032928466796875, 0.0036182...",0.504806
6,gray,"[0.004851989448070526, -0.02022680640220642, -...",0.503979
28,black,"[0.01116943359375, -0.012054443359375, -0.0193...",0.485274
38,brown,"[-0.03070068359375, -0.007198333740234375, 0.0...",0.481777
15,fox,"[-0.03349066898226738, 0.0015862904256209731, ...",0.473181


In [24]:
df

,text,embedding,similarities
0,sandwich,"[-0.00571441650390625, -0.048858642578125, -0....",0.207916
1,raccoon,"[0.05898793041706085, -0.0010472146095708013, ...",0.313692
2,milk,"[0.039642333984375, -0.004215240478515625, -0....",0.216576
3,green,"[0.006280624307692051, -0.0011062989942729473,...",0.458701
4,pizza,"[-0.010169987566769123, -0.028292447328567505,...",0.248450
5,lizard,"[0.01209514494985342, -0.012445430271327496, -...",0.298842
6,gray,"[0.004851989448070526, -0.02022680640220642, -...",0.503979
7,rabbit,"[0.006238460540771484, -0.01577582396566868, 0...",0.393911
8,guinea pig,"[0.0074920654296875, -0.0266571044921875, 0.00...",0.203466
9,cappuccino,"[-0.0248565673828125, -0.0309295654296875, -0....",0.183713


In [25]:
v1 = df['embedding'].iloc[2] # Milk
v2 = df['embedding'].iloc[9] # Cappucino
v = v1 + v2
df['similarities'] = df['embedding'].apply(lambda x: cosine_similarity(x, v))
df.sort_values('similarities', ascending=False).head(10)

,text,embedding,similarities
9,cappuccino,"[-0.0248565673828125, -0.0309295654296875, -0....",0.829932
2,milk,"[0.039642333984375, -0.004215240478515625, -0....",0.829802
19,coffee,"[-0.01006317138671875, 0.0037403106689453125, ...",0.507596
13,pasta,"[-0.05015381798148155, -0.04086881875991821, 0...",0.451773
39,soda,"[0.0184783935546875, -0.025054931640625, -0.03...",0.443256
27,tea,"[-0.016386866569519043, -0.030908560380339622,...",0.409103
0,sandwich,"[-0.00571441650390625, -0.048858642578125, -0....",0.378766
21,salad,"[-0.0007352828979492188, -0.037078857421875, 0...",0.343140
32,water,"[0.0030422210693359375, 0.0174560546875, -0.01...",0.335403
22,white,"[0.0032487320713698864, -0.02826567552983761, ...",0.334551
